# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_1_HANDS_ON_SESSION_3_HITL.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About this Notebook

Insert a Human-in-the-Loop Gate in Your Own Agent Workflow


It contains:
- Pattern A (LinkedIn-style direct interrupt loop)
- Pattern B (middleware-based tool gating)
- Policy edits, ordered decisions, and audit logging


In [1]:
!pip install -q langchain==1.2.12 langgraph==1.1.1 langchain-openai==1.0.3 python-dotenv==1.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 4.8 MB/s eta 0:00:00


In [2]:
# OpenAI-only setup
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not found in env.")
    OPENAI_API_KEY = input("Enter OPENAI_API_KEY: ").strip()

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"Using model: {OPENAI_MODEL}")


Using model: gpt-4.1-mini


In [3]:
from typing import Dict, Any, List, TypedDict
import copy
import uuid
from datetime import datetime, timezone

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.graph import StateGraph
from langgraph.constants import END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


## Pattern A - LinkedIn style multi-turn feedback loop (direct interrupt)


In [4]:
llm = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)

class DraftState(TypedDict, total=False):
    topic: str
    draft: str
    feedback_log: List[str]

def draft_node(state: DraftState) -> DraftState:
    fb = state.get("feedback_log", [])
    prompt = (
        f"Topic: {state['topic']}\n"
        f"Latest feedback: {fb[-1] if fb else 'No feedback yet'}\n\n"
        "Write a concise LinkedIn post. If feedback exists, revise accordingly."
    )
    text = llm.invoke(prompt).content
    print("\n[Draft]\n", text)
    return {"draft": text, "feedback_log": fb}

def review_node(state: DraftState):
    decision = interrupt(
        {
            "task": "Review this draft. Provide feedback text, or type done to finish.",
            "current_draft": state["draft"],
        }
    )
    if isinstance(decision, str) and decision.strip().lower() in {"done", "quit", "exit"}:
        return Command(goto="finish_node")
    return Command(
        goto="draft_node",
        update={"feedback_log": state.get("feedback_log", []) + [str(decision)]},
    )

def finish_node(state: DraftState) -> DraftState:
    print("\n[Final Draft]\n", state.get("draft", ""))
    return state

def build_feedback_graph():
    g = StateGraph(DraftState)
    g.add_node("draft_node", draft_node)
    g.add_node("review_node", review_node)
    g.add_node("finish_node", finish_node)
    g.set_entry_point("draft_node")
    g.add_edge("draft_node", "review_node")
    g.add_edge("finish_node", END)
    return g.compile(checkpointer=InMemorySaver())

def run_pattern_a(topic: str = "Human-in-the-loop in production AI agents"):
    app = build_feedback_graph()
    cfg = {"configurable": {"thread_id": f"pattern-a-{uuid.uuid4()}"}}
    result = app.invoke({"topic": topic, "feedback_log": []}, config=cfg)
    while getattr(result, "interrupts", None):
        intr = result.interrupts[0]
        print("\nInterrupt payload:")
        print(intr.value)
        fb = input("Feedback (or done): ").strip()
        result = app.invoke(Command(resume=fb), config=cfg)

# Example:
# run_pattern_a("Why HITL is important for enterprise agents")


## Pattern B - HITL middleware for sensitive tool calls


## Decision Cheat Sheet (Pattern B)

Suggested reviewer defaults:

- `record_human_feedback` -> **approve**
- `update_customer_plan` -> **edit** (validate arguments)
- `execute_sql` -> **reject** for non-`SELECT` queries
- `deactivate_customer_account` -> **reject** unless explicit human authorization exists

In production, the reviewer should make this decision explicitly each time based on risk and policy, not based on auto-suggestions.


## How to reason about interrupt decisions

Use this checklist every time a tool call is paused:

### Risk check
- Is the tool **read-only** or **state-changing**?
- Is the action reversible?
- Could this impact money, compliance, or user data?

### Decision guide
- **approve**: safe action + correct arguments.
- **edit**: intent is valid, but arguments need correction.
- **reject**: unsafe/out-of-policy action, missing context, or wrong target.

### Practical examples
- `update_customer_plan` -> often start with **edit** to verify `customer_id`, `new_plan`, `reason`.
- `execute_sql` -> **approve** only for clearly read-only `SELECT`; otherwise **reject** in this lab.
- `deactivate_customer_account` -> usually **reject** unless a documented approval exists.
- `record_human_feedback` -> usually **approve** (low-risk metadata update).

### Teaching goal
Interrupts are where you apply policy, not just mechanics. The point is to practice safe human judgment before execution.


In [5]:
# Mocked customer database
DB: Dict[str, Dict[str, Any]] = {
    "C001": {
        "name": "Ava Patel",
        "plan": "pro",
        "balance": 39.99,
        "active": True,
        "notes": ["Requested billing explanation last month"],
    },
    "C002": {
        "name": "Leo Kim",
        "plan": "basic",
        "balance": 0.0,
        "active": True,
        "notes": [],
    },
}

FEEDBACK_LOG: List[Dict[str, str]] = []
AUDIT_LOG: List[Dict[str, Any]] = []

@tool
def read_customer(customer_id: str) -> Dict[str, Any]:
    """Read customer data from the mocked database."""
    row = DB.get(customer_id)
    if not row:
        return {"error": f"Customer {customer_id} not found"}
    return copy.deepcopy(row)

@tool
def record_human_feedback(customer_id: str, feedback: str) -> str:
    """Store human reviewer feedback for a customer conversation."""
    FEEDBACK_LOG.append({"customer_id": customer_id, "feedback": feedback})
    return f"Stored feedback for {customer_id}: {feedback}"

@tool
def execute_sql(query: str) -> str:
    """Simulate a sensitive SQL execution without touching a real database."""
    return f"[MOCK SQL EXECUTED] {query}"

@tool
def update_customer_plan(customer_id: str, new_plan: str, reason: str) -> Dict[str, Any]:
    """Update a customer's subscription plan in the mocked database."""
    if customer_id not in DB:
        return {"ok": False, "error": f"Customer {customer_id} not found"}
    old = DB[customer_id]["plan"]
    DB[customer_id]["plan"] = new_plan
    DB[customer_id]["notes"].append(f"Plan changed from {old} to {new_plan}. Reason: {reason}")
    return {
        "ok": True,
        "customer_id": customer_id,
        "old_plan": old,
        "new_plan": new_plan,
    }

@tool
def deactivate_customer_account(customer_id: str, reason: str) -> Dict[str, Any]:
    """Deactivate a customer account (high-risk action)."""
    if customer_id not in DB:
        return {"ok": False, "error": f"Customer {customer_id} not found"}
    DB[customer_id]["active"] = False
    DB[customer_id]["notes"].append(f"Account deactivated. Reason: {reason}")
    return {"ok": True, "customer_id": customer_id, "active": DB[customer_id]["active"]}


### Why this policy configuration matters

This cell defines *what must pause* for review. In production, this is your safety contract:
- safe reads can run automatically,
- writes/deletes/financial operations should pause,
- decision options should match risk (`approve/reject` only for very risky actions).


In [6]:
# Baseline policy
checkpointer = InMemorySaver()

agent = create_agent(
    model=f"openai:{OPENAI_MODEL}",
    system_prompt=(
        "You are a customer-ops assistant. When a user asks for an operation that maps to a tool, "
        "you MUST call that tool immediately and not ask follow-up confirmation if required arguments "
        "are already present. Do not narrate intended actions; execute through tool calls. "
        "In multi-turn interactions, ask for human feedback before sensitive actions only when input "
        "is actually missing."
    ),
    tools=[read_customer, record_human_feedback, execute_sql, update_customer_plan, deactivate_customer_account],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_customer": False,
                "record_human_feedback": False,
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},
                "update_customer_plan": True,
                "deactivate_customer_account": {"allowed_decisions": ["approve", "reject"]},
            },
            description_prefix="Tool execution pending approval",
        )
    ],
    checkpointer=checkpointer,
)


### Reviewer behavior in production

This next cell intentionally pauses and asks the human reviewer to choose a decision manually for each sensitive action.

Use this quick decision map before typing anything:
- `record_human_feedback`: usually **approve** (low-risk metadata write).
- `update_customer_plan`: **edit** if arguments are imprecise, otherwise **approve** only when policy justification is present.
- `execute_sql`: **approve** only for read-only `SELECT`; **reject** any `INSERT/UPDATE/DELETE/DDL` in this lab.
- `deactivate_customer_account`: **approve** only with explicit security/compliance authorization reference; otherwise **reject**.

Why this matters:
- reviewers are accountable for high-risk actions,
- decisions must be explicit and policy-based,
- audit logs should reflect real human judgment, including reason text for rejections.


In [7]:
def print_interrupts(result) -> None:
    interrupts = getattr(result, "interrupts", None) or []
    if not interrupts:
        print("No interrupts.")
        return
    for i, intr in enumerate(interrupts, 1):
        print(f"\nInterrupt {i}:")
        print(intr.value)

def ask_decisions_for_interrupt(interrupt_value: Dict[str, Any]) -> Dict[str, Any]:
    actions: List[Dict[str, Any]] = interrupt_value.get("action_requests", [])
    review_cfgs: List[Dict[str, Any]] = interrupt_value.get("review_configs", [])
    decisions = []

    for idx, action in enumerate(actions):
        cfg = review_cfgs[idx] if idx < len(review_cfgs) else {}
        allowed = cfg.get("allowed_decisions", ["approve", "edit", "reject"])
        name = action.get("name")
        args = action.get("args") or action.get("arguments") or {}

        print("\n--- Review Action ---")
        print(f"Tool: {name}")
        print(f"Arguments: {args}")
        print(f"Allowed decisions: {allowed}")
        print("Reviewer checklist: target identity, scope, policy reason, reversibility")

        prompt = f"Decision ({'/'.join(allowed)}): "
        while True:
            decision_type = input(prompt).strip().lower()
            if decision_type in allowed:
                break
            print("Invalid decision for this tool. Try again.")

        if decision_type == "approve":
            decisions.append({"type": "approve"})
        elif decision_type == "reject":
            feedback = input("Reject reason shown to agent: ").strip() or "Rejected by human reviewer"
            decisions.append({"type": "reject", "reason": feedback})
        else:
            edited = dict(args)
            print("Current args:", edited)
            while True:
                k = input("Arg key to edit (blank to finish): ").strip()
                if not k:
                    break
                edited[k] = input(f"New value for {k}: ").strip()

            edited_action = dict(action)
            if "args" in edited_action:
                edited_action["args"] = edited
            else:
                edited_action["arguments"] = edited
            decisions.append({"type": "edit", "edited_action": edited_action})

    return {"decisions": decisions}


### Why we run with a stable thread id

In production, interrupts require persisted thread state so the run can pause and resume safely after human review. This runner cell models that behavior and records each review decision for auditability.


In [8]:
# Multi-turn runner with completed audit logging
baseline_cfg = {"configurable": {"thread_id": f"hitl-demo-{uuid.uuid4()}"}}
print("Baseline thread id:", baseline_cfg["configurable"]["thread_id"])

def _interrupt_action_names(result) -> List[str]:
    interrupts = getattr(result, "interrupts", None) or []
    names: List[str] = []
    for intr in interrupts:
        value = getattr(intr, "value", {}) or {}
        actions = value.get("action_requests", []) if isinstance(value, dict) else []
        names.extend(a.get("name") for a in actions if isinstance(a, dict) and a.get("name"))
    return names

def _invoke_with_expected_interrupts(
    agent_obj,
    cfg: Dict[str, Any],
    user_text: str,
    expected_tools: List[str] | None,
):
    result = agent_obj.invoke(
        {"messages": [{"role": "user", "content": user_text}]},
        config=cfg,
        version="v2",
    )

    if not expected_tools:
        return result

    action_names = _interrupt_action_names(result)
    missing = [t for t in expected_tools if t not in action_names]
    if not missing:
        return result

    # Retry once with an explicit tool-call instruction to reduce model variance.
    strict_prompt = (
        f"Tool-call correction: call these tools now: {expected_tools}. "
        "Do not ask follow-up questions and do not skip the tool call. "
        f"Original user request: {user_text}"
    )
    result = agent_obj.invoke(
        {"messages": [{"role": "user", "content": strict_prompt}]},
        config=cfg,
        version="v2",
    )

    action_names = _interrupt_action_names(result)
    missing = [t for t in expected_tools if t not in action_names]
    if missing:
        raise RuntimeError(
            f"Expected interrupt(s) for {missing}, but got {action_names or 'no interrupts'}"
        )
    return result

def run_turn(agent_obj, cfg: Dict[str, Any], user_text: str, expected_tools: List[str] | None = None):
    result = _invoke_with_expected_interrupts(agent_obj, cfg, user_text, expected_tools)

    while getattr(result, "interrupts", None):
        print_interrupts(result)
        intr = result.interrupts[0]
        resume_payload = ask_decisions_for_interrupt(intr.value)

        AUDIT_LOG.append(
            {
                "timestamp": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
                "thread_id": cfg["configurable"]["thread_id"],
                "interrupt": intr.value,
                "decision": resume_payload,
            }
        )

        result = agent_obj.invoke(Command(resume=resume_payload), config=cfg, version="v2")

    payload = getattr(result, "value", result)
    messages = payload.get("messages", []) if isinstance(payload, dict) else []
    print("\nAssistant response:")
    for msg in messages[-3:]:
        role = getattr(msg, "type", None) or msg.__class__.__name__
        text = getattr(msg, "content", "")
        print(f"- {role}: {text}")

def show_db():
    print("\nCurrent DB snapshot:")
    for cid, row in DB.items():
        print(cid, row)


Baseline thread id: hitl-demo-72f0e28e-aea6-48ba-a1c6-ec8cda2196ea


### Guided baseline walkthrough

In this section, you will act as the human reviewer in a customer-operations workflow.

What to pay attention to:
- read-only actions should usually pass without interruption,
- low-risk notes can often be recorded directly,
- high-impact account changes should pause for review,
- your job is to decide whether the agent should proceed, be corrected, or be blocked.

As you move through the next cells, read the markdown before each interrupt carefully. It tells you exactly what decision to choose, what to type, and why that choice reflects a realistic production policy for AI agents.


In [9]:
# Start the walkthrough with safe and low-risk actions.
show_db()
run_turn(agent, baseline_cfg, "Use read_customer for C001 and summarize status in one short paragraph.")
run_turn(
    agent,
    baseline_cfg,
    "Use record_human_feedback with customer_id='C001' and feedback='Customer requested downgrade review due budget constraints (ticket CS-1042)'.",
)



Current DB snapshot:
C001 {'name': 'Ava Patel', 'plan': 'pro', 'balance': 39.99, 'active': True, 'notes': ['Requested billing explanation last month']}
C002 {'name': 'Leo Kim', 'plan': 'basic', 'balance': 0.0, 'active': True, 'notes': []}

Assistant response:
- ai: 
- tool: {"name": "Ava Patel", "plan": "pro", "balance": 39.99, "active": true, "notes": ["Requested billing explanation last month"]}
- ai: Customer Ava Patel is currently subscribed to the pro plan with an active status and has a balance of $39.99. She had requested a billing explanation last month, which is noted in her account. Overall, her account is in good standing and active.

Assistant response:
- ai: 
- tool: Stored feedback for C001: Customer requested downgrade review due budget constraints (ticket CS-1042)
- ai: Feedback noting the customer's request for downgrade review due to budget constraints has been recorded for customer C001.


### Interrupt 1: customer plan change

The next cell asks the agent to downgrade `C001` from `pro` to `basic`. This should interrupt because subscription changes affect billing and customer experience, so a human should validate the request before the agent changes account state.

What to choose:
- Type `approve`.

Why:
- the customer id is specific,
- the requested plan is clear,
- the reason includes a supervisor-approved ticket reference,
- this is a realistic example of a state-changing action that is allowed only after human review.

What to watch for in the interrupt window:
- confirm the tool is `update_customer_plan`,
- confirm `customer_id='C001'`,
- confirm `new_plan='basic'`,
- confirm the reason still mentions the approval context.

Production lesson:
- even when an action is acceptable, the interrupt is still useful because it creates a review checkpoint before money-affecting changes are applied.


In [10]:
run_turn(
    agent,
    baseline_cfg,
    "Use update_customer_plan with customer_id='C001', new_plan='basic', reason='Budget downgrade approved by supervisor in ticket CS-1042'.",
    expected_tools=["update_customer_plan"],
)



Interrupt 1:
{'action_requests': [{'name': 'update_customer_plan', 'args': {'customer_id': 'C001', 'new_plan': 'basic', 'reason': 'Budget downgrade approved by supervisor in ticket CS-1042'}, 'description': "Tool execution pending approval\n\nTool: update_customer_plan\nArgs: {'customer_id': 'C001', 'new_plan': 'basic', 'reason': 'Budget downgrade approved by supervisor in ticket CS-1042'}"}], 'review_configs': [{'action_name': 'update_customer_plan', 'allowed_decisions': ['approve', 'edit', 'reject']}]}

--- Review Action ---
Tool: update_customer_plan
Arguments: {'customer_id': 'C001', 'new_plan': 'basic', 'reason': 'Budget downgrade approved by supervisor in ticket CS-1042'}
Allowed decisions: ['approve', 'edit', 'reject']
Reviewer checklist: target identity, scope, policy reason, reversibility
Decision (approve/edit/reject): approve

Assistant response:
- ai: 
- tool: {"ok": true, "customer_id": "C001", "old_plan": "pro", "new_plan": "basic"}
- ai: The subscription plan for custom

### Interrupt 2: destructive SQL request

The next cell asks the agent to run a `DELETE` statement. This should interrupt because destructive database operations are one of the clearest examples of actions that require human review in production AI systems.

What to choose:
- Type `reject`.
- When prompted for a reason, type: `Destructive SQL is not approved in this workflow. Open a DBA-reviewed cleanup task instead.`

Why:
- the query is not read-only,
- it could delete data at scale,
- it is not easily reversible,
- a production agent should not be trusted to execute broad deletion operations without a stronger approval process.

What to watch for in the interrupt window:
- the tool name should be `execute_sql`,
- the query should clearly contain `DELETE`,
- the allowed decisions should only be `approve` and `reject`, which reflects a stricter policy for dangerous operations.


In [11]:
run_turn(
    agent,
    baseline_cfg,
    "Use execute_sql with query='DELETE FROM customer_events WHERE created_at < NOW() - INTERVAL ''30 days'';'.",
    expected_tools=["execute_sql"],
)



Interrupt 1:
{'action_requests': [{'name': 'execute_sql', 'args': {'query': "DELETE FROM customer_events WHERE created_at < NOW() - INTERVAL '30 days';"}, 'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE FROM customer_events WHERE created_at < NOW() - INTERVAL \'30 days\';"}'}], 'review_configs': [{'action_name': 'execute_sql', 'allowed_decisions': ['approve', 'reject']}]}

--- Review Action ---
Tool: execute_sql
Arguments: {'query': "DELETE FROM customer_events WHERE created_at < NOW() - INTERVAL '30 days';"}
Allowed decisions: ['approve', 'reject']
Reviewer checklist: target identity, scope, policy reason, reversibility
Decision (approve/reject): reject
Reject reason shown to agent: Destructive SQL is not approved in this workflow. Open a DBA-reviewed cleanup task instead.

Assistant response:
- ai: 
- tool: User rejected the tool call for `execute_sql` with id call_VY8z0idnR0ehEvn0LBjyY468
- ai: The execution of the SQL query to delete

### Interrupt 3: high-risk account deactivation

The next cell asks the agent to deactivate a customer account. This is the kind of action that should almost always interrupt in production because it can lock a real user out of service and create legal or support consequences.

What to choose:
- Type `approve`.

Why:
- the request includes a concrete reason,
- it references explicit security authorization (`SEC-8821`),
- the target customer is identified clearly,
- this is exactly the kind of high-risk action that still needs a human sign-off even when the case looks legitimate.

What to watch for in the interrupt window:
- confirm the `customer_id` is `C002`,
- confirm the reason still includes the security approval reference,
- notice that this tool only allows `approve` or `reject`, not `edit`, because production systems often reduce reviewer freedom for the most sensitive actions.


In [12]:
run_turn(
    agent,
    baseline_cfg,
    "Use deactivate_customer_account with customer_id='C002' and reason='Confirmed account takeover; security approval SEC-8821 authorizes immediate deactivation'.",
    expected_tools=["deactivate_customer_account"],
)



Interrupt 1:
{'action_requests': [{'name': 'deactivate_customer_account', 'args': {'customer_id': 'C002', 'reason': 'Confirmed account takeover; security approval SEC-8821 authorizes immediate deactivation'}, 'description': "Tool execution pending approval\n\nTool: deactivate_customer_account\nArgs: {'customer_id': 'C002', 'reason': 'Confirmed account takeover; security approval SEC-8821 authorizes immediate deactivation'}"}], 'review_configs': [{'action_name': 'deactivate_customer_account', 'allowed_decisions': ['approve', 'reject']}]}

--- Review Action ---
Tool: deactivate_customer_account
Arguments: {'customer_id': 'C002', 'reason': 'Confirmed account takeover; security approval SEC-8821 authorizes immediate deactivation'}
Allowed decisions: ['approve', 'reject']
Reviewer checklist: target identity, scope, policy reason, reversibility
Decision (approve/reject): approve

Assistant response:
- ai: 
- tool: {"ok": true, "customer_id": "C002", "active": false}
- ai: Customer account C

### After the interrupts: verify the outcome

Run the next cell after you finish all three reviews.

What to look for:
- the feedback log should contain the earlier human note,
- `C001` should show the approved plan change,
- the SQL cleanup should **not** have changed the mock database because you rejected it,
- `C002` should be inactive only if you approved the final security action.

This final check reinforces a core production habit: do not stop at the decision screen. Always verify the downstream system state after a sensitive action is approved or rejected.


In [13]:
print("\nFeedback log:", FEEDBACK_LOG)
show_db()



Feedback log: [{'customer_id': 'C001', 'feedback': 'Customer requested downgrade review due budget constraints (ticket CS-1042)'}]

Current DB snapshot:
C001 {'name': 'Ava Patel', 'plan': 'basic', 'balance': 39.99, 'active': True, 'notes': ['Requested billing explanation last month', 'Plan changed from pro to basic. Reason: Budget downgrade approved by supervisor in ticket CS-1042']}
C002 {'name': 'Leo Kim', 'plan': 'basic', 'balance': 0.0, 'active': False, 'notes': ['Account deactivated. Reason: Confirmed account takeover; security approval SEC-8821 authorizes immediate deactivation']}


## Policy edit task

In this step, the policy becomes stricter.

What changed:
- `record_human_feedback` now also interrupts for review,
- `update_customer_plan` no longer allows `edit`, only `approve` or `reject`.

Why this matters in production:
- even seemingly small notes can shape future agent behavior,
- some organizations do not let reviewers silently fix risky requests,
- when a sensitive action arrives with weak evidence, the correct action is often to reject it and send it back for a cleaner request.

The next cells walk you through that stricter review posture one decision at a time.


In [14]:
agent_step3_solution = create_agent(
    model=f"openai:{OPENAI_MODEL}",
    system_prompt=(
        "You are a customer-ops assistant. Use tools whenever a request maps to available tools. "
        "Do not ask follow-up confirmation if arguments are already provided."
    ),
    tools=[read_customer, record_human_feedback, execute_sql, update_customer_plan, deactivate_customer_account],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_customer": False,
                "record_human_feedback": True,
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},
                "update_customer_plan": {"allowed_decisions": ["approve", "reject"]},
                "deactivate_customer_account": {"allowed_decisions": ["approve", "reject"]},
            },
            description_prefix="Tool execution pending approval",
        )
    ],
    checkpointer=InMemorySaver(),
)

step3_cfg = {"configurable": {"thread_id": f"step3-{uuid.uuid4()}"}}
print("Step 3 thread id:", step3_cfg["configurable"]["thread_id"])


Step 3 thread id: step3-4f089a71-3048-4bd0-b758-23b572031b22


### Interrupt 1: reviewed feedback capture

In this stricter policy, even `record_human_feedback` now interrupts. That may feel conservative, but it reflects a real production concern: reviewer notes can influence future agent decisions, so organizations may require oversight even for metadata writes.

What to choose:
- Type `approve`.

Why:
- the note is narrow and policy-oriented,
- it does not directly change the account,
- the request is clear enough to allow,
- this shows how a company can require review for guidance that may later affect automation.

What to watch for:
- confirm the tool is `record_human_feedback`,
- confirm the note is attached to `C001`,
- confirm the text is a constrained policy note rather than a direct account change.


In [15]:
run_turn(
    agent_step3_solution,
    step3_cfg,
    "Use record_human_feedback with customer_id='C001' and feedback='Enterprise upgrades require documented legal approval before execution'.",
    expected_tools=["record_human_feedback"],
)



Interrupt 1:
{'action_requests': [{'name': 'record_human_feedback', 'args': {'customer_id': 'C001', 'feedback': 'Enterprise upgrades require documented legal approval before execution'}, 'description': "Tool execution pending approval\n\nTool: record_human_feedback\nArgs: {'customer_id': 'C001', 'feedback': 'Enterprise upgrades require documented legal approval before execution'}"}], 'review_configs': [{'action_name': 'record_human_feedback', 'allowed_decisions': ['approve', 'edit', 'reject']}]}

--- Review Action ---
Tool: record_human_feedback
Arguments: {'customer_id': 'C001', 'feedback': 'Enterprise upgrades require documented legal approval before execution'}
Allowed decisions: ['approve', 'edit', 'reject']
Reviewer checklist: target identity, scope, policy reason, reversibility
Decision (approve/edit/reject): approve

Assistant response:
- ai: 
- tool: Stored feedback for C001: Enterprise upgrades require documented legal approval before execution
- ai: The feedback regarding "E

### Interrupt 2: no `edit` allowed on plan changes

The next cell requests an enterprise upgrade, but the evidence is too weak: the reason only says `Legal approved` and gives no ticket, case id, or approval reference. Under this stricter policy, the reviewer is not allowed to repair the request by editing it.

What to choose:
- Type `reject`.
- When prompted for a reason, type: `Missing approval evidence. Resubmit with a legal ticket or documented authorization reference.`

Why:
- the request changes subscription state,
- the justification is incomplete,
- `edit` is intentionally disabled for this tool,
- in production, some teams prefer rejection over reviewer rewriting so the original requester remains accountable for the final instruction.

What to watch for:
- confirm the tool is `update_customer_plan`,
- notice that allowed decisions are only `approve` and `reject`,
- confirm there is not enough approval context to support an enterprise upgrade.


In [16]:
run_turn(
    agent_step3_solution,
    step3_cfg,
    "Use update_customer_plan with customer_id='C001', new_plan='enterprise', reason='Legal approved'.",
    expected_tools=["update_customer_plan"],
)



Interrupt 1:
{'action_requests': [{'name': 'update_customer_plan', 'args': {'customer_id': 'C001', 'new_plan': 'enterprise', 'reason': 'Legal approved'}, 'description': "Tool execution pending approval\n\nTool: update_customer_plan\nArgs: {'customer_id': 'C001', 'new_plan': 'enterprise', 'reason': 'Legal approved'}"}], 'review_configs': [{'action_name': 'update_customer_plan', 'allowed_decisions': ['approve', 'reject']}]}

--- Review Action ---
Tool: update_customer_plan
Arguments: {'customer_id': 'C001', 'new_plan': 'enterprise', 'reason': 'Legal approved'}
Allowed decisions: ['approve', 'reject']
Reviewer checklist: target identity, scope, policy reason, reversibility
Decision (approve/reject): reject
Reject reason shown to agent: Missing approval evidence. Resubmit with a legal ticket or documented authorization reference.

Assistant response:
- ai: 
- tool: User rejected the tool call for `update_customer_plan` with id call_Ez3O3Hysufl90raLjQyrT3K5
- ai: Sorry, it seems there was 

### Interrupt 3: default-deny on weak deactivation requests

The next cell asks to deactivate `C002` for a `Compliance hold`, but it does not include a case number, approval reference, or incident identifier. That is not enough evidence for a shutdown action.

What to choose:
- Type `reject`.
- When prompted for a reason, type: `Missing compliance authorization reference. Do not deactivate until a documented case id is provided.`

Why:
- deactivation is a high-impact action,
- the reason is too vague,
- there is no explicit authority attached to the request,
- strong production HITL policies usually default to deny when the evidence trail is incomplete.

What to watch for:
- confirm the tool is `deactivate_customer_account`,
- confirm the target is `C002`,
- confirm the reason lacks a specific approval reference,
- notice again that only `approve` or `reject` are allowed for this risk level.


In [17]:
run_turn(
    agent_step3_solution,
    step3_cfg,
    "Use deactivate_customer_account with customer_id='C002' and reason='Compliance hold'.",
    expected_tools=["deactivate_customer_account"],
)



Interrupt 1:
{'action_requests': [{'name': 'deactivate_customer_account', 'args': {'customer_id': 'C002', 'reason': 'Compliance hold'}, 'description': "Tool execution pending approval\n\nTool: deactivate_customer_account\nArgs: {'customer_id': 'C002', 'reason': 'Compliance hold'}"}], 'review_configs': [{'action_name': 'deactivate_customer_account', 'allowed_decisions': ['approve', 'reject']}]}

--- Review Action ---
Tool: deactivate_customer_account
Arguments: {'customer_id': 'C002', 'reason': 'Compliance hold'}
Allowed decisions: ['approve', 'reject']
Reviewer checklist: target identity, scope, policy reason, reversibility
Decision (approve/reject): reject
Reject reason shown to agent: Missing compliance authorization reference. Do not deactivate until a documented case id is provided.

Assistant response:
- ai: 
- tool: User rejected the tool call for `deactivate_customer_account` with id call_4eg3rDsE4X1TJunh2eJo1dXq
- ai: I attempted to deactivate the account for customer ID C002 

### Verify what stricter policy changed

After the three Step 3 reviews, check the outcome carefully.

What you should observe:
- the reviewed feedback note was allowed,
- the enterprise plan change was blocked,
- the account deactivation was blocked,
- stricter policy changed not only which tools interrupt, but also what the reviewer is allowed to do once interrupted.

This is an important production design lesson: HITL is not only about pausing execution. It is also about constraining reviewer actions to match organizational risk policy.


In [18]:
print("\nFeedback log after Step 3:", FEEDBACK_LOG)
show_db()



Feedback log after Step 3: [{'customer_id': 'C001', 'feedback': 'Customer requested downgrade review due budget constraints (ticket CS-1042)'}, {'customer_id': 'C001', 'feedback': 'Enterprise upgrades require documented legal approval before execution'}]

Current DB snapshot:
C001 {'name': 'Ava Patel', 'plan': 'basic', 'balance': 39.99, 'active': True, 'notes': ['Requested billing explanation last month', 'Plan changed from pro to basic. Reason: Budget downgrade approved by supervisor in ticket CS-1042']}
C002 {'name': 'Leo Kim', 'plan': 'basic', 'balance': 0.0, 'active': False, 'notes': ['Account deactivated. Reason: Confirmed account takeover; security approval SEC-8821 authorizes immediate deactivation']}


## Ordered decisions

This step teaches a subtle but important HITL rule: when one interrupt contains multiple pending actions, your decisions must line up with the actions in exactly the same order.

Why this matters in production:
- one human review event may contain multiple tool calls,
- the reviewer might approve one action but reject another,
- if decision order is mismatched, the wrong action can be approved or blocked,
- that is a real operational risk when actions affect billing, data, or account access.

The next cells walk you through this as a tightly guided multi-action review.


In [19]:
ordered_cfg = {"configurable": {"thread_id": f"ordered-{uuid.uuid4()}"}}
ordered_result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Update C001 to basic because supervisor ticket CS-2041 approved the downgrade, and also delete old customer events in the same turn.",
            }
        ]
    },
    config=ordered_cfg,
    version="v2",
)
print("Interrupts:", ordered_result.interrupts)


Interrupts: (Interrupt(value={'action_requests': [{'name': 'update_customer_plan', 'args': {'customer_id': 'C001', 'new_plan': 'basic', 'reason': 'supervisor ticket CS-2041 approved the downgrade'}, 'description': "Tool execution pending approval\n\nTool: update_customer_plan\nArgs: {'customer_id': 'C001', 'new_plan': 'basic', 'reason': 'supervisor ticket CS-2041 approved the downgrade'}"}, {'name': 'execute_sql', 'args': {'query': "DELETE FROM customer_events WHERE customer_id = 'C001'"}, 'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE FROM customer_events WHERE customer_id = \'C001\'"}'}], 'review_configs': [{'action_name': 'update_customer_plan', 'allowed_decisions': ['approve', 'edit', 'reject']}, {'action_name': 'execute_sql', 'allowed_decisions': ['approve', 'reject']}]}, id='5fe5014cc92d5045824c76ced0f54554'),)


### Review: apply decisions in the same order as the actions

The previous cell should show one interrupt containing multiple action requests. Read the interrupt payload carefully and keep the action order exactly as shown.

What to choose for the first action:
- If the first action is `update_customer_plan`, treat it as allowed but tighten the justification.
- The decision should be `edit` with these arguments:
  - `customer_id`: `C001`
  - `new_plan`: `basic`
  - `reason`: `Supervisor-approved downgrade under ticket CS-2041`

What to choose for the second action:
- If the second action is `execute_sql`, choose `reject`.
- Use this reject reason: `Destructive cleanup requires DBA approval and cannot be bundled with a customer account change.`

Why this ordering matters:
- the first decision applies to the first action request,
- the second decision applies to the second action request,
- if you swap them, you can accidentally reject the safe action and approve the dangerous one,
- that is exactly why ordered decision payloads matter in production HITL systems.


In [20]:
# Decision order must match action_requests order in the interrupt payload.
first_interrupt = ordered_result.interrupts[0]
action_requests = (first_interrupt.value or {}).get("action_requests", [])
ordered_decisions = []

for action in action_requests:
    name = action.get("name")
    if name == "update_customer_plan":
        edited_action = dict(action)
        original_args = action.get("args") or action.get("arguments") or {}
        edited_args = {
            **original_args,
            "customer_id": "C001",
            "new_plan": "basic",
            "reason": "Supervisor-approved downgrade under ticket CS-2041",
        }
        if "args" in edited_action:
            edited_action["args"] = edited_args
        else:
            edited_action["arguments"] = edited_args
        ordered_decisions.append({"type": "edit", "edited_action": edited_action})
    elif name == "execute_sql":
        ordered_decisions.append(
            {
                "type": "reject",
                "reason": "Destructive cleanup requires DBA approval and cannot be bundled with a customer account change.",
            }
        )
    else:
        ordered_decisions.append({"type": "reject", "reason": f"Unexpected action: {name}"})

decisions = {"decisions": ordered_decisions}
ordered_final = agent.invoke(Command(resume=decisions), config=ordered_cfg, version="v2")
print("Ordered flow completed:", ordered_final)


Ordered flow completed: GraphOutput(value={'messages': [HumanMessage(content='Update C001 to basic because supervisor ticket CS-2041 approved the downgrade, and also delete old customer events in the same turn.', additional_kwargs={}, response_metadata={}, id='c4050c77-fed4-4bb4-977e-c25586f81ea0'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 265, 'total_tokens': 341, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c5385b4086', 'id': 'chatcmpl-DNAhGFa2ghBR7vey8N8hdDjX63EZf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2360-b80a-73d2-a930-5d5355f41238-0', tool_calls=[{'type': 'tool_call', 'name': 'up

### Takeaway

After resuming, confirm the outcome conceptually:
- the plan-change action should proceed with the corrected reason,
- the SQL cleanup should remain blocked,
- the decision list worked only because it matched the interrupt action order.

Production lesson:
- multi-action interrupts save reviewer time,
- but they also increase the risk of human error,
- so reviewers must inspect ordering carefully before resuming the agent.


## State inspection and optional state edit

This section shows what a human operator or control-plane service can do after an interrupt is raised but before the run is resumed.

Why this matters in production:
- sometimes the reviewer needs more context than the interrupt window shows,
- persisted state lets you inspect the paused run safely,
- some teams also inject manual override guidance before resuming,
- the final decision should still reflect the actual risk of the pending actions.


In [21]:
manual_cfg = {"configurable": {"thread_id": "manual-hitl-demo-solutions"}}
manual_result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Update customer C001 to enterprise for legal reasons and run SQL cleanup on old customer events.",
            }
        ]
    },
    config=manual_cfg,
    version="v2",
)
print("Interrupts:", manual_result.interrupts)

state = agent.get_state(manual_cfg)
print("State keys:", list((state.values or {}).keys()))


Interrupts: (Interrupt(value={'action_requests': [{'name': 'update_customer_plan', 'args': {'customer_id': 'C001', 'new_plan': 'enterprise', 'reason': 'legal reasons'}, 'description': "Tool execution pending approval\n\nTool: update_customer_plan\nArgs: {'customer_id': 'C001', 'new_plan': 'enterprise', 'reason': 'legal reasons'}"}, {'name': 'execute_sql', 'args': {'query': "DELETE FROM customer_events WHERE customer_id = 'C001' AND event_date < CURRENT_DATE - INTERVAL '1 year'"}, 'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE FROM customer_events WHERE customer_id = \'C001\' AND event_date < CURRENT_DATE - INTERVAL \'1 year\'"}'}], 'review_configs': [{'action_name': 'update_customer_plan', 'allowed_decisions': ['approve', 'edit', 'reject']}, {'action_name': 'execute_sql', 'allowed_decisions': ['approve', 'reject']}]}, id='26f200aec16bcb04046106d94d4400ed'),)
State keys: ['messages']


### What to inspect before resuming

In a real system, you would not resume immediately. First inspect the paused state.

What to look for:
- confirm the run really paused on the thread you expect,
- inspect the state keys to see what context is persisted,
- verify which actions are pending before any human override is applied.

Why this matters:
- state inspection helps prevent approving the wrong paused run,
- it gives operators a way to audit context before resuming,
- it is especially useful when many agent runs may be paused at the same time.


### Resume with a conservative manual decision set

For this paused run, the request is intentionally weak: it asks for an enterprise plan change and SQL cleanup without documented approval references.

What the next cell will do:
- reject `update_customer_plan` because the justification is not specific enough,
- reject `execute_sql` because destructive cleanup needs stronger approval,
- build the resume payload in the same order as the pending action list.

Production lesson:
- state inspection is useful only if it changes the decision quality,
- the human or control-plane logic should translate that context into an explicit, risk-aware resume payload.


In [22]:
# Optional (version-dependent):
# agent.update_state(
#     manual_cfg,
#     {"messages": [{"role": "system", "content": "Human override: prioritize conservative actions."}]},
# )

first_interrupt = manual_result.interrupts[0]
action_requests = (first_interrupt.value or {}).get("action_requests", [])
manual_decisions = []

for action in action_requests:
    name = action.get("name")
    if name == "update_customer_plan":
        manual_decisions.append(
            {
                "type": "reject",
                "reason": "Missing documented approval for enterprise upgrade. Provide a legal or finance authorization reference.",
            }
        )
    elif name == "execute_sql":
        manual_decisions.append(
            {
                "type": "reject",
                "reason": "SQL cleanup requires DBA-reviewed change control and cannot be approved from this request.",
            }
        )
    else:
        manual_decisions.append({"type": "reject", "reason": "Rejected by conservative manual review."})

resume_payload = {"decisions": manual_decisions or [{"type": "reject", "reason": "No approved actions."}]}

final = agent.invoke(Command(resume=resume_payload), config=manual_cfg, version="v2")
print("Manual flow completed:", final)


Manual flow completed: GraphOutput(value={'messages': [HumanMessage(content='Update customer C001 to enterprise for legal reasons and run SQL cleanup on old customer events.', additional_kwargs={}, response_metadata={}, id='52132dbe-a237-4800-8023-8ef860ebcb07'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 256, 'total_tokens': 337, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c5385b4086', 'id': 'chatcmpl-DNAhJjrafKhEeTcpPjvROapF7DUke', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2360-c1ef-7431-98f4-4b6c38cda165-0', tool_calls=[{'name': 'update_customer_plan', 'args': {'customer_id': 'C001', 'new_

### State-inspection takeaway

Notice what this section demonstrates:
- the run can be paused and inspected safely,
- the reviewer does not need to decide blindly,
- resumed decisions can be generated programmatically from the inspected action list,
- the final outcome can still reflect a conservative human policy.


## Streaming

This section shows the same HITL pattern during streaming output.

Why this matters in production:
- users may see tokens stream before the agent reaches a gated action,
- the system still has to pause safely at the interrupt boundary,
- a reviewer or controller can then resume the stream with explicit decisions.


In [23]:
stream_cfg = {"configurable": {"thread_id": f"stream-{uuid.uuid4()}"}}
pending_interrupt = None

for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete stale records and then downgrade C001 to basic because supervisor ticket CS-3108 approved the downgrade.",
            }
        ]
    },
    config=stream_cfg,
    stream_mode=["updates", "messages"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, _metadata = chunk["data"]
        if getattr(token, "content", None):
            print(token.content, end="", flush=True)
    elif chunk["type"] == "updates":
        if "__interrupt__" in chunk["data"]:
            pending_interrupt = chunk["data"]["__interrupt__"]
            print("\n\nInterrupt:", pending_interrupt)




Interrupt: (Interrupt(value={'action_requests': [{'name': 'execute_sql', 'args': {'query': "DELETE FROM records WHERE status = 'stale'"}, 'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE FROM records WHERE status = \'stale\'"}'}, {'name': 'update_customer_plan', 'args': {'customer_id': 'C001', 'new_plan': 'basic', 'reason': 'Supervisor ticket CS-3108 approved the downgrade'}, 'description': "Tool execution pending approval\n\nTool: update_customer_plan\nArgs: {'customer_id': 'C001', 'new_plan': 'basic', 'reason': 'Supervisor ticket CS-3108 approved the downgrade'}"}], 'review_configs': [{'action_name': 'execute_sql', 'allowed_decisions': ['approve', 'reject']}, {'action_name': 'update_customer_plan', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='8d4be211a01dd4f499693df3d8c4a763'),)


### What to watch during streaming

Run the previous cell and watch for two things:
- normal token output may appear first,
- then the stream pauses when the interrupt is emitted.

That pause is the key production behavior: streaming does not bypass human review. The agent can still be stopped exactly when it reaches a sensitive tool call.


### Resume the stream with risk-aware decisions

The next cell resumes the stream programmatically.

What it will do:
- reject `execute_sql` because the request is destructive,
- approve `update_customer_plan` when the action includes the supervisor ticket reference,
- keep the decision order aligned with the streamed interrupt payload.

Why this is useful in production:
- a backend controller can apply the same safety policy automatically after a human review step,
- streaming UX remains responsive while still enforcing interrupt-based governance.


In [24]:
stream_decisions = {"decisions": [{"type": "reject", "reason": "No interrupt payload found."}]}
if pending_interrupt:
    first = pending_interrupt[0] if isinstance(pending_interrupt, (list, tuple)) else pending_interrupt
    payload = getattr(first, "value", first)
    action_requests = (payload or {}).get("action_requests", []) if isinstance(payload, dict) else []
    ordered_decisions = []

    for action in action_requests:
        name = action.get("name")
        if name == "execute_sql":
            ordered_decisions.append(
                {
                    "type": "reject",
                    "reason": "Destructive SQL requires DBA approval and is blocked in the streaming workflow.",
                }
            )
        elif name == "update_customer_plan":
            ordered_decisions.append({"type": "approve"})
        else:
            ordered_decisions.append({"type": "reject", "reason": "Unhandled action in streaming review."})

    stream_decisions = {"decisions": ordered_decisions or [{"type": "reject", "reason": "No actions to approve."}]}

for chunk in agent.stream(
    Command(resume=stream_decisions),
    config=stream_cfg,
    stream_mode=["updates", "messages"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, _metadata = chunk["data"]
        if getattr(token, "content", None):
            print(token.content, end="", flush=True)


User rejected the tool call for `execute_sql` with id call_XIILWQHw1pcOewCOB03BF33w{"ok": true, "customer_id": "C001", "old_plan": "basic", "new_plan": "basic"}I have downgraded customer C001 to the basic plan as approved by the supervisor ticket CS-3108. However, I was unable to delete stale records because I need your confirmation to perform this database operation. Please confirm if you want me to proceed with deleting stale records.

### Streaming takeaway

After the stream resumes, the downgrade path can continue while the SQL cleanup remains blocked.

Production lesson:
- streaming and HITL are compatible,
- you can preserve a good user experience without giving up control over sensitive actions,
- the important part is that the interrupt payload is inspected and resumed with explicit policy-aware decisions.


## Audit review

In production, review logs are evidence of human oversight.

What this final section reinforces:
- every interrupt should leave a trace,
- every human decision should be reviewable later,
- timestamps and decision payloads matter for compliance, debugging, and post-incident analysis.


In [25]:
print(f"Audit entries: {len(AUDIT_LOG)}")
for i, entry in enumerate(AUDIT_LOG, 1):
    print(f"\nAudit #{i}")
    print("Decision:", entry.get("decision"))
    print("Timestamp:", entry.get("timestamp"))


Audit entries: 6

Audit #1
Decision: {'decisions': [{'type': 'approve'}]}
Timestamp: 2026-03-25T05:02:26.406276Z

Audit #2
Decision: {'decisions': [{'type': 'reject', 'reason': 'Destructive SQL is not approved in this workflow. Open a DBA-reviewed cleanup task instead.'}]}
Timestamp: 2026-03-25T05:02:50.613209Z

Audit #3
Decision: {'decisions': [{'type': 'approve'}]}
Timestamp: 2026-03-25T05:02:58.310044Z

Audit #4
Decision: {'decisions': [{'type': 'approve'}]}
Timestamp: 2026-03-25T05:03:13.232561Z

Audit #5
Decision: {'decisions': [{'type': 'reject', 'reason': 'Missing approval evidence. Resubmit with a legal ticket or documented authorization reference.'}]}
Timestamp: 2026-03-25T05:03:34.342129Z

Audit #6
Decision: {'decisions': [{'type': 'reject', 'reason': 'Missing compliance authorization reference. Do not deactivate until a documented case id is provided.'}]}
Timestamp: 2026-03-25T05:03:53.131132Z
